In [1]:
# ====================== 环境准备 ======================
# 1) 设定环境变量（必须在导入 matplotlib 之前执行）
import os

from draw.basic_functio.get_rectangular_size_interval import calc_envelope_for_group

os.environ['QT_API'] = 'pyqt5'        # 指定使用 PyQt5 作为 Qt 绑定
os.environ['MPLBACKEND'] = 'QtAgg'    # 指定 Matplotlib 后端为 QtAgg（更推荐，替代 TkAgg）

# 2) 启动 Qt 事件循环（控制台模式下，让 Qt 窗口能实时响应）
%gui qt5

# 3) 检查 matplotlib backend
import matplotlib as mpl
mpl.rcParams.update({
    # 这里按系统常见字体给一串候选，存在则自动生效
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "SimSun",
                        "Noto Sans CJK SC", "Source Han Sans SC",
                        "Arial Unicode MS", "DejaVu Sans"],
    "font.family": "sans-serif",
    "axes.unicode_minus": False,   # 负号用正常字符，避免被当作缺字形
})

print("backend (before pyplot):", mpl.get_backend())
# 如果不是 QtAgg，强制改为 QtAgg（注意：必须在导入 pyplot 前设置）
mpl.rcParams['backend'] = 'QtAgg'

# 4) 现在再导入 pyplot
import matplotlib.pyplot as plt
print("backend (after pyplot):", mpl.get_backend())
from config import DATA_DIR,INPUT_DIR
from typing import Dict, Any

def slice_group_data(raw_group_data, start, end):
    """
    从 raw_group_data 中裁剪时间区间 [basicSa, end)
    """
    return {
        step: raw_group_data[step]
        for step in range(start, end)
        if step in raw_group_data
    }







backend (before pyplot): module://matplotlib_inline.backend_inline
backend (after pyplot): QtAgg


In [2]:
from pathlib import Path
# 默认用 "topology_{TIME_2_BUILD}"，也允许用环境变量 TOPOLOGY_VERSION 覆盖
VERSION = os.getenv("TOPOLOGY_VERSION", f"baseline")

RAW_DIR    = Path(INPUT_DIR) / VERSION / "raw"
CONFIG_DIR = Path(INPUT_DIR) /VERSION / "baselie"


In [3]:
def ensure_dirs(*paths: Path):
    for p in paths:
        try:
            p.mkdir(parents=True, exist_ok=True)
        except FileExistsError:
            # 目录名已被一个同名“文件”占用
            if not p.is_dir():
                raise NotADirectoryError(f"存在同名文件，无法创建目录: {p}")
        except Exception as e:
            raise RuntimeError(f"创建目录失败 {p}: {e}")

# 保证目录存在（多进程下也安全、可重复调用）
ensure_dirs(RAW_DIR, CONFIG_DIR)
# ====================== 导入依赖 ======================
import sys
# 避免反复执行时 Qt 类重复导入导致崩溃：如果已加载，先删除再导入
if 'draw.pyqt_draw.pyqt_main2' in sys.modules:
    del sys.modules['draw.pyqt_draw.pyqt_main2']

from PyQt5 import QtWidgets
import pyqtgraph as pg
from draw.pyqt_draw.pyqt_main2 import SatelliteViewer
import draw.read_snap_xml  as read_snap_xml
import draw.read_snap_xml  as read_snap_xml
# 配置 pyqtgraph：开启抗锯齿，关闭 OpenGL（更稳定）
pg.setConfigOptions(antialias=True)

In [4]:
file_in = DATA_DIR
# xml_file = r"DATA_DIR\station_visible_satellites_648_1d_real.xml"
xml_file = DATA_DIR / "station_visible_satellites_648_1d_real.xml"



In [5]:
# ====================== 基础参数 ======================
# 星座参数：每轨道卫星数 N，轨道平面数 P
N = 36
P = 18

# ====================== 读取数据 ======================

# start_ts = 10717
# # end_ts   = 86399
# end_ts   = 11640
# # 解析 XML 得到 group_data，结构：{time_step: {'groups': {...}}}
# group_data = read_snap_xml.parse_xml_group_data(xml_file, start_ts, end_ts)
# 只做一次：解析大区间
RAW_START, RAW_END = 0, 30152
raw_group_data = read_snap_xml.parse_xml_group_data(xml_file, RAW_START, RAW_END)

#下面是图变换的。
#

In [6]:

# 用法（左闭右开
start_ts =0

end_ts =22005

group_data = slice_group_data(raw_group_data, start_ts, end_ts)


In [11]:
base_groupid_now = 4

In [8]:
rev_group_data,offset = read_snap_xml.modify_group_data(group_data,P=18, N=36, base_groupid=base_groupid_now)

下面主要是为了测试检测我们的图

In [7]:

# ====================== 绘图初始化 ======================
# 1) QApplication 实例（全局唯一）
app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])

# 2) 确保 viewer 有全局引用，避免 GC 回收导致崩溃
if not hasattr(sys.modules[__name__], "_viewer_list"):
    _viewer_list = []

In [8]:


# 3) 创建并配置 viewer
viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("group_data")
viewer.resize(1200, 700)
# viewer.edges_by_step =
# viewer.pending_links_by_step =
viewer.show()



In [ ]:
# 3) 创建并配置 viewer
viewer = SatelliteViewer(rev_group_data)
viewer.setWindowTitle("rev_group_data")
viewer.resize(1200, 700)
# viewer.edges_by_step =
# viewer.pending_links_by_step =
viewer.show()


In [11]:
# 4) 保存全局引用,只要放入到这个容器里，就能持久存在
_viewer_list.append(viewer)

下面两段代码，分别是研发和对比试验，通过这样的操作，我们能顾极大的减少我们研究的步骤


下面是同构图验证，只需要查看一个拓扑图即可，

这里有一个额外的代码，那就是温和过渡。
所谓的温和过渡，就是在某些情况下，按照上述老方法，会出现，一口气切换一大块链路，但实际上，没必要一口气切换这么大的链路，因为会造成网络的不稳定性。

下述的代码，一般情况下是不需要运行的，一般是到后面进行微调才运行的


In [12]:
import draw.basic_functio.topology_config as topology_config

rec = topology_config.TopologyRecorder(P, N)
rec.modify_group_data(group_data, base_groupid=base_groupid_now)

nodes = {}


#永久设置，
rec.write_distinct_motif(0, 17, 0, 35, nodes, option=0)

# rec.write_distinct_motif(0, 6, 20, 27, nodes, option=0)
#
# rec.write_distinct_motif(0, 6, 20, 30, nodes, option=0)
# rec.write_distinct_motif(6, 17, 19, 28, nodes, option=1)
# rec.write_distinct_motif(0, 4, 31, 35, nodes, option=0)
# rec.write_distinct_motif(4, 6, 31, 35, nodes, option=2)
# rec.write_distinct_motif(6, 17, 29, 35, nodes, option=0)

#
# rec.write_distinct_motif_with_time(
#     6, 17, 28, 28, nodes,
#     t_start=f"start_ts + time_2_build",
#     t_end  =f"end_ts  ",
#     option=0,
#
# )
#
#
#
# max_motif_up = 9
#
#
# for motif_up in range(0, max_motif_up + 1):
#     k = motif_up + 1
#     rec.write_distinct_motif_with_time(
#         6, 7, 20, 20+motif_up, nodes,
#         t_start=f"16940 + time_2_build*{k}",
#         t_end  =f"16940 + time_2_build*{k+1}",
#         option=0,
#
#     )
# rec.write_distinct_motif_with_time(
#     6, 7, 20, 19+motif_up, nodes,
#         t_start=f"16940 + time_2_build*{k}",
#         t_end  =f"end_ts",
#     option=0,
#
# )


#14528


# 保存时“symbols”可选（只是提示配置里用到了哪些符号）
rec.save(CONFIG_DIR / f"{start_ts}_{end_ts}.json",
         symbols=["start_ts", "end_ts", "time_2_build"])
env = {
    "start_ts": start_ts,
    "end_ts": end_ts,

}






all_rev_inter_edge = rec.render_adj_range(start_ts, end_ts, eval_env=env)

In [ ]:
# 这里是grid的设置，一般情况下是用不到的

In [13]:
viewer = SatelliteViewer(rev_group_data)
viewer.setWindowTitle("rev_group_data with rev_group_data")
viewer.resize(1200, 700)
viewer.edges_by_step = all_rev_inter_edge

viewer.show()
_viewer_list.append(viewer)
# viewer.show_envelopes_static(
#     rects_by_group=rects,
#     expand=0.35,
#     colors=colors,
#     persist=True
# )

NameError: name 'rev_group_data' is not defined

In [ ]:
import draw.basic_functio.topology_config as topology_config

cfg = topology_config.load_config(CONFIG_DIR / f"{start_ts}_{end_ts}.json")

# 渲染时提供变量环境（可以动态变更 time_2_build 等）
env = {
    "start_ts": start_ts,
    "end_ts": end_ts,
    "time_2_build": time_2_build,
}

rec = topology_config.TopologyRecorder(cfg.P, cfg.N)
rec.base_groupid = cfg.base_groupid
rec._motifs = cfg.motifs

# 渲染某一秒的邻接
# adj_1232 = rec.render_adj_at(1232, eval_env=env)

# 渲染整段并生成 all_rev_inter_edge（你的老变量名）
all_rev_inter_edge = rec.render_adj_range(start_ts, end_ts, eval_env=env)


In [ ]:

# 3) 下面是同构图设计，我们一般从同构图上设计出 motif，然后，再迁移到我们其他图的显示上去
import draw.pyqt_draw.pyqt_onetopology as pyqt_onetopology
viewer = pyqt_onetopology.Onetopology(rects)
viewer.setWindowTitle("Grouped Satellite Visibility - High Performance (PyQtGraph)")
viewer.resize(1200, 700)
motif_construct_edge = rev_inter_edge
viewer.edges_by_step = motif_construct_edge
viewer.show()


In [ ]:
viewer = SatelliteViewer(rev_group_data)
viewer.setWindowTitle("rev_group_data with rev_group_data")
viewer.resize(1200, 700)
viewer.edges_by_step = all_rev_inter_edge

viewer.show()
_viewer_list.append(viewer)

### 处理冲突链路
上述获得的是原始图的边，但是，其存在冲突问题，例如 t=100s是a与b t=101s就是a与c了，考虑到链路建链需要时间，因此，我们这里需要进行一些冲突修正
假设建链时间是setuptime  =60s
1. 若a与b是区域内链路，a与c是是区域外链路，由于区域内链路是必须保留的，因此，我们认为，a与c的链路是从t=101才开始建链，并且t=160末尾才完成建链，并在t=161时投入使用
2. 若a与b是区域外链路，a与c是区域内链路，由于区域内链路是必须保留的，因此，我们认为，a与c的链路在t=101时刻就可投入使用，因此，a与b的链路已经在前60s断链，认为a与c的链路在t=101-60也就是y=41s时开始建链



In [14]:

# 注意上述我们是在同构拓扑序列上进行的，因此，我们还要将同构拓扑序列进行还原，同时，我们还要考虑到建链时间约束
import draw.basic_functio.revdata2rawdata as revdata2rawdata
# attention ,here  it just composed of the inter-link, intra_link hasn't benn conclued
raw_inter_edge = revdata2rawdata.revedge2rawedge(all_rev_inter_edge,offset)

In [16]:


#xiamianshi meiyouyiyi de
viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("groupdata with rawedge")

viewer.resize(1200, 700)
viewer.edges_by_step =raw_inter_edge

viewer.show()


In [ ]:
# 4) 保存全局引用,只要放入到这个容器里，就能持久存在
_viewer_list.append(viewer)

我们要处理好建链时间冲突，因此，下面就是处理冲突的代码

注意，这里其实就是IG 内部要处理的建联图

In [17]:

# 下面是把边转为node存储，因为这种方式存储会比较方便


import draw.basic_functio.inter_edge2nodes as inter_edge2nodes
all_nodes = inter_edge2nodes.trans_edge2node(raw_inter_edge,P,N)

1


In [18]:
##在检查实际拓扑后，确认无问题后，就将其存入到xml文件里去，注意，我们只要保存异轨链路信息即可，其余不必保存
import draw.basic_functio.write2xml as write2xml
import genaric2.tegnode as tegnode
# 推荐：用 raw string 防止反斜杠转义，并改成有意义的文件名
# 这里，我们要把原始的边转为node进行存储


file_path = RAW_DIR / f"interplane_links_{start_ts}_{end_ts}.xml"


write2xml.nodes_to_xml(
    all_nodes,
   file_path
)


True
